In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from Configs.Spark_Core.session import get_spark_session,create_namespaces
from Configs.Spark_Core import tables

spark=get_spark_session()



In [0]:
spark.sql("""
INSERT INTO AstroSight.bronze.api_endpoints
VALUES (
    1,
    'NASA',
    'neo',
    'https://api.nasa.gov/neo/rest/v1/feed',
    '{"start_date":"today","end_date":"today"}',
    'Y',
    'Y',
    'scheduled'
)
""")
spark.sql("""
INSERT INTO AstroSight.bronze.api_endpoints
VALUES (
    2,
    'NASA',
    'gst',
    'https://api.nasa.gov/DONKI/GST',
    '{"start_date":"today","end_date":"today"}',
    'Y',
    'Y',
    'scheduled'
)
""")
spark.sql("""
INSERT INTO AstroSight.bronze.api_endpoints
VALUES (
    3,
    'NASA',
    'apod',
    'https://api.nasa.gov/planetary/apod',
    '{"start_date":"today","end_date":"today"}',
    'Y',
    'Y',
    'scheduled'
)
""")
spark.sql("""
INSERT INTO AstroSight.bronze.api_endpoints
VALUES (
    4,
    'NASA',
    'cme',
    'https://api.nasa.gov/DONKI/CME',
    '{"start_date":"today","end_date":"today"}',
    'Y',
    'Y',
    'scheduled'
)
""")
spark.sql("""
INSERT INTO AstroSight.bronze.api_endpoints
VALUES (
    5,
    'NASA',
    'IPS',
    'https://api.nasa.gov/DONKI/IPS',
    '{"start_date":"today","end_date":"today"}',
    'Y',
    'Y',
    'scheduled'
)
""")


In [0]:
from Spark.Extract.NeoWS_Extract import execute_scheduled_requests

spark.table("astrosight.bronze.api_response").show()

In [0]:


spark.sql(f"""
    INSERT INTO AstroSight.config.api_table_mapping
VALUES
('cme',4,'silver','cme_ids',1,'$[*]',NULL,'Y',CURRENT_TIMESTAMP,'N'),
('cme',4,'silver','cme_analysis',2,'$[*]','cmeAnalyses','Y',CURRENT_TIMESTAMP,'Y'),
('cme',4,'silver','cme_instruments',3,'$[*]','instruments','Y',CURRENT_TIMESTAMP,'Y')
""")


spark.sql(f"""
INSERT INTO AstroSight.config.api_column_mapping
VALUES
('cme','cme_ids','cme_id','activityID','STRING',1,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_catalog','catalog','STRING',2,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_starttime','startTime','TIMESTAMP',3,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_sourcelocation','sourceLocation','STRING',4,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_submissiontime','submissionTime','TIMESTAMP',5,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_versionid','versionId','INT',6,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_note','note','STRING',7,'Y',CURRENT_TIMESTAMP),
('cme','cme_ids','cme_link','link','STRING',8,'Y',CURRENT_TIMESTAMP)
""")\


spark.sql(f"""
INSERT INTO AstroSight.config.api_column_mapping
VALUES
('cme','cme_analysis','analysis_id',NULL,'STRING',1,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','cme_id','ROOT.activityID','STRING',2,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','is_most_accurate','isMostAccurate','BOOLEAN',3,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','time21_5','time21_5','TIMESTAMP',4,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','latitude','latitude','DOUBLE',5,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','longitude','longitude','DOUBLE',6,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','halfAngle','halfAngle','DOUBLE',7,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','speed','speed','DOUBLE',8,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','type','type','STRING',9,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','featureCode','featureCode','STRING',10,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','levelOfData','levelOfData','INT',11,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','tilt','tilt','DOUBLE',12,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','speedMeasuredAtHeight','speedMeasuredAtHeight','DOUBLE',13,'Y',CURRENT_TIMESTAMP),
('cme','cme_analysis','submissionTime','submissionTime','TIMESTAMP',14,'Y',CURRENT_TIMESTAMP)
""")

spark.sql(f"""
INSERT INTO AstroSight.config.api_column_mapping
VALUES
('cme','cme_instruments','instrument_id',NULL,'UUID',1,'Y',CURRENT_TIMESTAMP),
('cme','cme_instruments','cme_id','ROOT.activityID','STRING',2,'Y',CURRENT_TIMESTAMP),
('cme','cme_instruments','instrument_recorded','displayName','STRING',3,'Y',CURRENT_TIMESTAMP)
""")

spark.sql(f"""
INSERT INTO AstroSight.config.table_merge_mapping
VALUES
('cme_ids','silver','merge_into_cme_ids'),
('cme_analysis','silver','merge_into_cme_analysis'),
('cme_instruments','silver','merge_into_cme_instruments')
""")

spark.sql("UPDATE AstroSight.config.api_table_mapping set is_context_array='N' where target_table in ('cme_ids')")

spark.sql("UPDATE AstroSight.config.api_table_mapping set is_context_array='Y' where target_table in ('cme_analysis','cme_instruments')")

spark.sql("UPDATE AstroSight.config.api_column_mapping t set json_source_path='UUID',data_type='STRING' where t.target_column in ('instrument_id','analysis_id')")



# -----------------

# --IPS--

# spark.sql("""
# INSERT INTO AstroSight.bronze.api_endpoints
# VALUES (
#     5,
#     'NASA',
#     'IPS',
#     'https://api.nasa.gov/DONKI/IPS',
#     '{"start_date":"today","end_date":"today"}',
#     'Y',
#     'Y',
#     'scheduled'
# )
# """)

spark.sql("""
INSERT INTO AstroSight.config.api_table_mapping
VALUES
('IPS',5, 'silver', 'ips_ids',1, '$[*]', NULL,'Y', CURRENT_TIMESTAMP, 'N'),
('IPS',5, 'silver', 'ips_instruments', 2, '$[*]', 'instruments', 'Y', CURRENT_TIMESTAMP, 'Y')
""")

spark.sql("""
INSERT INTO AstroSight.config.api_column_mapping
VALUES
('IPS','ips_ids','ips_id','activityID','STRING',1,'Y',CURRENT_TIMESTAMP),
('IPS','ips_ids','ips_catalog','catalog','STRING',2,'Y',CURRENT_TIMESTAMP),
('IPS','ips_ids','ips_location','location','STRING',3,'Y',CURRENT_TIMESTAMP),
('IPS','ips_ids','ips_eventtime','eventTime','TIMESTAMP',4,'Y',CURRENT_TIMESTAMP),
('IPS','ips_ids','ips_submissiontime','submissionTime','TIMESTAMP',5,'Y',CURRENT_TIMESTAMP),
('IPS','ips_ids','ips_versionid','versionId','INT',6,'Y',CURRENT_TIMESTAMP),
('IPS','ips_ids','ips_link','link','STRING',7,'Y',CURRENT_TIMESTAMP)
""")

spark.sql("""
INSERT INTO AstroSight.config.api_column_mapping
(api_name, target_table, target_column, json_source_path,
 data_type, column_order, is_active, ingestion_timestamp)
VALUES
('IPS','ips_instruments','ips_instrument_id',NULL,'UUID',1,'Y',CURRENT_TIMESTAMP),
('IPS','ips_instruments','ips_id','activityID','STRING',1,'Y',CURRENT_TIMESTAMP),
('IPS','ips_instruments','instrument_recorded','displayName','STRING',2,'Y',CURRENT_TIMESTAMP)
""")

spark.sql("UPDATE AstroSight.config.api_column_mapping set json_source_path='UUID',data_type='STRING' where target_table='ips_instruments' and target_column='ips_instrument_id'")

spark.sql("UPDATE AstroSight.config.api_column_mapping set json_source_path='ROOT.activityID' where target_table='ips_instruments' and target_column='ips_id'")

spark.sql(f"""
INSERT INTO AstroSight.config.table_merge_mapping
VALUES
('ips_ids','silver','merge_into_ips_ids'),
('ips_instruments','silver','merge_into_ips_instruments')
""")

In [0]:
from Configs.Spark_Core import session,insertion,tables
from Spark.Transform import Neo_Tarnsformer,Neo_Approach_Transformer,Gst_Kp_Transformer,Gst_Transformer,apod_details_transformer,CME_Transformer,IPS_Transformer
from Spark.Load import Neo_Rankings_Load,Neo_Summary_Load
from Spark.Load.DAYN import GST_Rankings_DAYN,GST_Summary_DAYN,Neo_Summary_Load_DAYN,Neo_Rankings_Load_DAYN,CME_Activity_Score_DAYN,CME_SUMMARY_DAYN
from Spark.Load.DAY0 import GST_Summary_DAY0,Neo_Rankings_DAY0,Neo_Summary_Load_DAY0
from Configs.AWS import S3_TO_Bronze


def Neo(spark):
    passed_ids_1 = Neo_Tarnsformer.transform_neo_data()
    passed_ids_2 = Neo_Approach_Transformer.transform_neo_close_data()
    passed_ids = list(set(passed_ids_1).intersection(set(passed_ids_2)))

    failed_ids = list(set(passed_ids_1).symmetric_difference(set(passed_ids_2)))
    try:
        if passed_ids:
            insertion.Update_api_response_status(passed_ids, spark)
        if failed_ids:
            insertion.Mark_api_response_as_failed(failed_ids,spark)
    except Exception as e:
        print(f"Error updating API response status: {e}")
    Neo_Summary_Load_DAYN.neo_summary_load_DAYN()
    Neo_Rankings_Load_DAYN.Neo_Rankings()

def GST(spark):
    passed_ids_1 = Gst_Transformer.gst_transform_data()
    passed_ids_2 = Gst_Kp_Transformer.gst_kp_details()
    passed_ids = list(set(passed_ids_1).intersection(set(passed_ids_2)))
    failed_ids = list(set(passed_ids_1).symmetric_difference(set(passed_ids_2)))

    try:
        if passed_ids:
            insertion.Update_api_response_status(passed_ids, spark)
        if failed_ids:
            insertion.Mark_api_response_as_failed(failed_ids,spark)
    except Exception as e:
        print(f"Error updating API response status: {e}")
    GST_Rankings_DAYN.gst_Rankings_DAYN()
    GST_Summary_DAYN.gst_summary_DAY0()


def APOD(spark):
    passed_ids = apod_details_transformer.apod_details_transform()
    insertion.Update_api_response_status(request_id=passed_ids,spark=spark)

def CME(spark):
    passed_ids = CME_Transformer.transform_cme_data()
    insertion.Update_api_response_status(request_id=passed_ids,spark=spark)
    CME_Activity_Score_DAYN.cme_activity_score_DAYN()
    CME_SUMMARY_DAYN.cme_summary_DAYN()

def IPS(spark):
    passed_ids = IPS_Transformer.transform_ips_data()
    insertion.Update_api_response_status(request_id=passed_ids,spark=spark)

Neo(spark)
GST(spark)
APOD(spark)
CME(spark=spark)
IPS(spark=spark)